In [ ]:
import pandas as pd

In [ ]:
raw_df = pd.read_csv("drx dataset.csv")

# Keep only records with the fields used in the modelling workflow.
# The source CSV also contains Atmosphere-only metadata rows, which are excluded here.
required_columns = ["Composition", "DRX label", "Method", "t (h)", "T (°C)"]
initial_row_count = len(raw_df)
drx_df = raw_df.dropna(subset=required_columns).loc[:, required_columns].copy()
print(f"Loaded {initial_row_count} source rows; removed {initial_row_count - len(drx_df)} incomplete rows; {len(drx_df)} modelling records remain.")
drx_df

In [ ]:
print(drx_df.dtypes)

In [ ]:
drx_df["DRX label"] = drx_df["DRX label"].str.lower()

In [ ]:
drx_df["DRX label"]

In [ ]:
drx_df["DRX label"] = drx_df["DRX label"].map({"successful": 1, "unsuccessful": 0})
# map(...) → 按照括号里的对照表替换值

In [ ]:
drx_df["DRX label"]

In [ ]:
print(drx_df["Method"].value_counts())
print(drx_df["t (h)"].value_counts())

In [ ]:
drx_df = drx_df.rename(columns={"Method\n": "Method"})

In [ ]:
drx_df["Method"] = pd.Categorical(drx_df["Method"], categories=["Hydrothermal", "Mechanochemical", "Molten-salt", "Sol-gel", "Solid-state"])
#pd.Categorical(...) pandas里的一个功能，把普通的一列数据变成category类型。括号里放两个东西：第一个是你的数据，第二个是categories=你自己指定类别。

method_categories = drx_df["Method"].cat.categories
#把这个["Hydrothermal", "Mechanochemical", "Molten-salt", "Sol-gel", "Solid-state"]列表存进一个叫method_categories的变量里

drx_df["Method"] = drx_df["Method"].cat.codes
#把类别名字按顺序换成数字

In [ ]:
drx_df["Method"]

In [ ]:
drx_df["t (h)"] = pd.to_numeric(drx_df["t (h)"], errors="coerce")

In [ ]:
print((drx_df["Method"] == -1).sum())
drx_df["t (h)"].isna().sum()
# Check category coding and missing synthesis-time values.

In [ ]:
drx_df[drx_df["t (h)"].isna()]

In [ ]:
# 温度和时间的基本统计：mean, min, max, range
print("=== T (°C) ===")
print(drx_df["T (°C)"].describe())

print()
print("=== t (h) ===")
print(drx_df["t (h)"].describe())

print()
print("=== DRX label ===")
print(drx_df["DRX label"].value_counts())

The final cleaned dataset contains 173 literature records and 115 unique composition formulae.
There are 144 successful records and 29 unsuccessful records (83.2% and 16.8%, respectively).
The class imbalance should be considered during model training and evaluation.

In [ ]:
from collections import Counter
import re #regular expression

# 把所有Composition里的元素提取出来
all_elements = []

for composition in drx_df["Composition"]:
    # re.findall找出所有大写字母开头的元素符号
    elements = re.findall(r'[A-Z][a-z]?', str(composition))
    all_elements.extend(elements)

# 数每个元素出现几次
element_counts = Counter(all_elements)
print(element_counts)

In [ ]:
# 把Composition和DRX label两列取出来，改成formula和target
cbfv_df = drx_df[["Composition", "DRX label"]].copy()
cbfv_df = cbfv_df.rename(columns={"Composition": "formula", "DRX label": "target"})
cbfv_df

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade setuptools

In [ ]:
!{sys.executable} -m pip show setuptools

In [ ]:
import sys
!{sys.executable} -m pip install "setuptools<81"

In [ ]:
import pandas as pd
from CBFV import composition

In [ ]:
x, y, formulae, skipped = composition.generate_features(cbfv_df)

In [ ]:
x

In [ ]:
num_rows = len(cbfv_df)
print(f'There are in total {num_rows} rows in the X_train DataFrame.')

num_unique_formulae = len(cbfv_df['formula'].unique())
print(f'But there are only {num_unique_formulae} unique formulae!\n')

In [ ]:
X = drx_df.drop(columns=["DRX label"]).copy()
y = drx_df["DRX label"].copy()

unique_formulae = X["Composition"].unique()

print(f"{len(unique_formulae)} unique formulae:\n{unique_formulae}")

In [ ]:
import json

# Reuse the exact composition-level split used for the dissertation results.
# This frozen manifest preserves the original 80/23/12 formula split.
with open("data_splits/split_manifest.json", "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

train_formulae = split_manifest["train_formulae"]
val_formulae = split_manifest["validation_formulae"]
test_formulae = split_manifest["test_formulae"]

train_formula_set = set(train_formulae)
val_formula_set = set(val_formulae)
test_formula_set = set(test_formulae)

if train_formula_set & val_formula_set or train_formula_set & test_formula_set or val_formula_set & test_formula_set:
    raise ValueError("Formula overlap detected between data splits.")

all_split_formulae = train_formula_set | val_formula_set | test_formula_set
if all_split_formulae != set(unique_formulae):
    missing = set(unique_formulae) - all_split_formulae
    unexpected = all_split_formulae - set(unique_formulae)
    raise ValueError(f"Split manifest does not match the dataset. Missing: {missing}; unexpected: {unexpected}")

print("Number of training formulae:", len(train_formulae))
print("Number of validation formulae:", len(val_formulae))
print("Number of testing formulae:", len(test_formulae))

In [ ]:
# Split the original dataset into train / validation / test datasets
# using the Composition lists above

df_train = drx_df[drx_df["Composition"].isin(train_formulae)].copy()
df_val = drx_df[drx_df["Composition"].isin(val_formulae)].copy()
df_test = drx_df[drx_df["Composition"].isin(test_formulae)].copy()

print(f"train dataset shape: {df_train.shape}")
print(f"validation dataset shape: {df_val.shape}")
print(f"test dataset shape: {df_test.shape}\n")

print(df_train.head(), "\n")
print(df_val.head(), "\n")
print(df_test.head(), "\n")

In [ ]:
train_formulae = set(df_train['Composition'].unique())
val_formulae = set(df_val['Composition'].unique())
test_formulae = set(df_test['Composition'].unique())

common_formulae1 = train_formulae.intersection(test_formulae)
common_formulae2 = train_formulae.intersection(val_formulae)
common_formulae3 = test_formulae.intersection(val_formulae)

print(f'# of common formulae in intersection 1: {len(common_formulae1)}; common formulae: {common_formulae1}')
print(f'# of common formulae in intersection 2: {len(common_formulae2)}; common formulae: {common_formulae2}')
print(f'# of common formulae in intersection 3: {len(common_formulae3)}; common formulae: {common_formulae3}')

In [ ]:
import os

# Save these splits into csv files
PATH = os.getcwd()

# Create a folder to store the split datasets
split_dir = os.path.join(PATH, "data_splits")
os.makedirs(split_dir, exist_ok=True)

train_path = os.path.join(split_dir, "drx_train.csv")
val_path = os.path.join(split_dir, "drx_val.csv")
test_path = os.path.join(split_dir, "drx_test.csv")

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print("Saved train data to:", train_path)
print("Saved validation data to:", val_path)
print("Saved test data to:", test_path)

In [ ]:
import numpy as np 


# =====================================================
# 0. Load the saved split files
# =====================================================

df_train = pd.read_csv("data_splits/drx_train.csv")
df_val = pd.read_csv("data_splits/drx_val.csv")
df_test = pd.read_csv("data_splits/drx_test.csv")

print("Loaded saved split files:")
print("df_train:", df_train.shape)
print("df_val:", df_val.shape)
print("df_test:", df_test.shape)


In [ ]:

# =====================================================
# 0.1 Featurise each split
# =====================================================

def featurise_split(df_split):
    cbfv_df = df_split[["Composition", "DRX label"]].copy()

    cbfv_df = cbfv_df.rename(
        columns={
            "Composition": "formula",
            "DRX label": "target"
        }
    )
# using oliynyk
    X_split, y_split, formulae_split, skipped_split = composition.generate_features(cbfv_df)

    X_split = X_split.reset_index(drop=True)
    y_split = pd.Series(y_split).reset_index(drop=True)

    return X_split, y_split, formulae_split, skipped_split


X_train, y_train, formulae_train, skipped_train = featurise_split(df_train)
X_val, y_val, formulae_val, skipped_val = featurise_split(df_val)
X_test, y_test, formulae_test, skipped_test = featurise_split(df_test)

print("Original X_train shape:", X_train.shape)
print("Original X_val shape:", X_val.shape)
print("Original X_test shape:", X_test.shape)


In [ ]:
# =====================================================
# 0.2 Start from the newly featurised split data
# =====================================================

X_train_filtered = X_train.copy()
X_val_filtered = X_val.copy()
X_test_filtered = X_test.copy()


# Make sure y_train has the same index as X_train
y_train_for_filter = pd.Series(y_train).copy()

if len(y_train_for_filter) != len(X_train_filtered):
    raise ValueError("y_train length is not the same as X_train length.")

# If the index does not match, force it to match X_train
if not y_train_for_filter.index.equals(X_train_filtered.index):
    y_train_for_filter.index = X_train_filtered.index



In [ ]:
# =====================================================
# 1. Remove low variance features
# =====================================================

variance_threshold = 0.0

numeric_cols = X_train_filtered.select_dtypes(include=["number"]).columns

x_num = X_train_filtered[numeric_cols].apply(pd.to_numeric, errors="coerce")

feature_variance = x_num.var(axis=0, skipna=True)

low_variance_features = []

for col in numeric_cols:
    v = float(feature_variance.get(col, np.nan))
    
    if (not np.isfinite(v)) or (v <= variance_threshold):
        low_variance_features.append(col)

# Drop the same low-variance features from train and test
X_train_filtered = X_train_filtered.drop(columns=low_variance_features, errors="ignore")
X_val_filtered = X_val_filtered.drop(columns=low_variance_features, errors="ignore")
X_test_filtered = X_test_filtered.drop(columns=low_variance_features, errors="ignore")

print("Low variance features removed:", len(low_variance_features))


In [ ]:
# =====================================================
# 2. Remove high correlation features
# =====================================================

correlation_threshold = 0.98

numeric_cols = X_train_filtered.select_dtypes(include=["number"]).columns

high_corr_records = []
high_correlation_features = []

if len(numeric_cols) >= 2:
    
    x_num = X_train_filtered[numeric_cols].apply(pd.to_numeric, errors="coerce")
    
    # Fill missing values using median from training data
    med = x_num.median(axis=0, skipna=True)
    x_num = x_num.fillna(med).fillna(0.0)
    
    # Calculate absolute correlation between features
    corr_abs = x_num.corr().abs().fillna(0.0)
    
    # Only use the upper triangle to avoid duplicate pairs
    upper = corr_abs.where(
        np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
    )
    
    # Calculate correlation between each feature and target
    y_num = pd.to_numeric(y_train_for_filter.reindex(x_num.index), errors="coerce")
    
    target_corr = {}
    
    for col in numeric_cols:
        xv = x_num[col].to_numpy(dtype=float)
        yv = y_num.to_numpy(dtype=float)
        
        mask = np.isfinite(xv) & np.isfinite(yv)
        
        if (
            int(np.sum(mask)) < 3
            or float(np.std(xv[mask])) < 1e-12
            or float(np.std(yv[mask])) < 1e-12
        ):
            target_corr[col] = float("nan")
        else:
            target_corr[col] = float(abs(np.corrcoef(xv[mask], yv[mask])[0, 1]))
    
    to_drop = set()
    sorted_pairs = upper.stack().sort_values(ascending=False)
    
    for (a, b), c in sorted_pairs.items():
        c_val = float(c)
        
        if c_val < correlation_threshold:
            break
        
        if a in to_drop or b in to_drop:
            continue
        
        ca = target_corr.get(str(a), float("nan"))
        cb = target_corr.get(str(b), float("nan"))
        
        # Keep the feature that has stronger correlation with the target
        if np.isnan(ca) and np.isnan(cb):
            drop_col = str(max(str(a), str(b)))
            keep_col = str(min(str(a), str(b)))
        elif np.isnan(ca):
            drop_col = str(a)
            keep_col = str(b)
        elif np.isnan(cb):
            drop_col = str(b)
            keep_col = str(a)
        elif ca < cb:
            drop_col = str(a)
            keep_col = str(b)
        elif cb < ca:
            drop_col = str(b)
            keep_col = str(a)
        else:
            drop_col = str(max(str(a), str(b)))
            keep_col = str(min(str(a), str(b)))
        
        to_drop.add(drop_col)
        
        high_corr_records.append({
            "feature_a": str(a),
            "feature_b": str(b),
            "abs_corr_train": c_val,
            "corr_with_target_a": ca,
            "corr_with_target_b": cb,
            "dropped_feature": drop_col,
            "kept_feature": keep_col,
            "threshold": correlation_threshold
        })
    
    high_correlation_features = sorted(to_drop)
    
    # Drop the same high-correlation features from train and test
    X_train_filtered = X_train_filtered.drop(columns=high_correlation_features, errors="ignore")
    X_val_filtered = X_val_filtered.drop(columns=high_correlation_features, errors="ignore")
    X_test_filtered = X_test_filtered.drop(columns=high_correlation_features, errors="ignore")

print("High correlation features removed:", len(high_correlation_features))


In [ ]:
# =====================================================
# 3. Make tables showing what was removed
# =====================================================

low_variance_removed = pd.DataFrame({
    "feature": low_variance_features,
    "train_variance": [feature_variance[f] for f in low_variance_features],
    "threshold": variance_threshold
})

high_correlation_removed = pd.DataFrame(high_corr_records)


# =====================================================
# 4. Save results
# =====================================================

low_variance_removed.to_csv("low_variance_removed.csv", index=False)
high_correlation_removed.to_csv("high_correlation_removed.csv", index=False)

X_train_filtered.to_csv("X_train_filtered.csv", index=False)
X_test_filtered.to_csv("X_test_filtered.csv", index=False)

print("Original X_train shape:", X_train.shape)
print("Filtered X_train shape:", X_train_filtered.shape)
print("Original X_test shape:", X_test.shape)
print("Filtered X_test shape:", X_test_filtered.shape)
print("Original X_val shape:", X_val.shape)
print("Filtered X_val shape:", X_val_filtered.shape)

In [ ]:
high_correlation_removed

In [ ]:
print(X_train_filtered.shape)
print(X_val_filtered.shape)
print(X_test_filtered.shape)

In [ ]:
X_train_filtered.to_csv("X_train_filtered.csv", index=False)
X_val_filtered.to_csv("X_val_filtered.csv", index=False)
X_test_filtered.to_csv("X_test_filtered.csv", index=False)